# Investigate Netztransparenz aFRR API Response

Goal: verify whether `99999` sentinel values appear in the raw API response and analyze statistical properties of aFRR data.

In [6]:
import os
from pathlib import Path

import polars as pl
import requests
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception as e:
    print(f'dotenv not available or failed to load: {e}')

sns.set_theme(style='whitegrid')

DATA_NETZ = Path('data/raw/netztransparenz.parquet')
DATA_ALL = Path('data/processed/all_data.parquet')
DATA_NETZ.exists(), DATA_ALL.exists()

(True, True)

## Part 1: Raw API Forensics (Single Request)

In [8]:
def _ensure_bearer(token: str) -> str:
    return token if token.startswith('Bearer ') else f'Bearer {token}'

def _fetch_token_from_client_credentials(client_id: str, client_secret: str) -> str:
    url = 'https://identity.netztransparenz.de/users/connect/token'
    payload = {
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
    }
    headers = {'Content-Type': 'application/x-www-form-urlencoded'}
    resp = requests.post(url, data=payload, headers=headers, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    access_token = data.get('access_token')
    if not access_token:
        raise RuntimeError('Token response missing access_token')
    return _ensure_bearer(access_token)

def _fetch_text(url: str, session: requests.Session, timeout: int = 60) -> str:
    resp = session.get(url, timeout=timeout)
    try:
        resp.raise_for_status()
    except requests.HTTPError as exc:
        detail = resp.text[:500] if resp.text else 'no response body'
        raise RuntimeError(f'Netztransparenz request failed ({resp.status_code}): {detail}') from exc
    return resp.text

base_url = 'https://ds.netztransparenz.de/api/v1/data'
start = '2022-12-12T00:00:00'
end = '2022-12-12T23:59:00'

token = os.getenv('NETZTRANSPARENZ_TOKEN')
if not token:
    client_id = os.getenv('NETZTRANSPARENZ_CLIENT_ID')
    client_secret = os.getenv('NETZTRANSPARENZ_CLIENT_SECRET')
    if client_id and client_secret:
        token = _fetch_token_from_client_credentials(client_id, client_secret)

if not token:
    print('Missing credentials: set NETZTRANSPARENZ_TOKEN or NETZTRANSPARENZ_CLIENT_ID/NETZTRANSPARENZ_CLIENT_SECRET')
else:
    session = requests.Session()
    session.headers.update({'Authorization': _ensure_bearer(token)})
    url = f'{base_url}/NrvSaldo/AktivierteSRL/Qualitaetsgesichert/{start}/{end}'
    raw_text = _fetch_text(url, session, timeout=60)
    print(raw_text[:2000])
    print('\nContains 99999:', '99999' in raw_text or '99.999' in raw_text)

Datum;Zeitzone;von;bis;Datenkategorie;Datentyp;Einheit;50Hertz (Positiv);Amprion (Positiv);TenneT TSO (Positiv);TransnetBW (Positiv);Deutschland (Positiv);50Hertz (Negativ);Amprion (Negativ);TenneT TSO (Negativ);TransnetBW (Negativ);Deutschland (Negativ)
12.12.2022;UTC;00:00;00:15;Aktivierte SRL;Qualitätsgesichert;MW;19,080;2,288;4,532;16,240;42,140;0,000;0,000;0,000;0,000;0,000
12.12.2022;UTC;00:15;00:30;Aktivierte SRL;Qualitätsgesichert;MW;0,004;0,056;0,000;0,008;0,068;0,000;0,000;0,000;2,772;2,772
12.12.2022;UTC;00:30;00:45;Aktivierte SRL;Qualitätsgesichert;MW;0,000;0,000;4,440;9,844;14,284;0,000;0,000;0,000;0,000;0,000
12.12.2022;UTC;00:45;01:00;Aktivierte SRL;Qualitätsgesichert;MW;0,000;0,000;0,000;0,000;0,000;0,000;0,000;0,000;0,560;0,560
12.12.2022;UTC;01:00;01:15;Aktivierte SRL;Qualitätsgesichert;MW;16,244;2,164;3,108;26,052;47,568;0,000;0,000;0,000;0,000;0,000
12.12.2022;UTC;01:15;01:30;Aktivierte SRL;Qualitätsgesichert;MW;14,952;2,272;6,264;20,188;43,676;0,000;0,000;0,000;0,0

## Part 2: Statistical Deep Dive (Full Dataset)

In [9]:
if DATA_NETZ.exists():
    data_path = DATA_NETZ
elif DATA_ALL.exists():
    data_path = DATA_ALL
else:
    raise FileNotFoundError('No netztransparenz.parquet or all_data.parquet found')

df = pl.read_parquet(data_path)
df.shape

(44570, 16)

In [10]:
cols = [
    'afrr_activation_price_pos',
    'afrr_activation_price_neg',
    'afrr_activation_avg_price_pos',
    'afrr_activation_avg_price_neg',
    'afrr_activated_mw_pos',
    'afrr_activated_mw_neg',
]
missing = [c for c in cols if c not in df.columns]
if missing:
    raise KeyError(f'Missing columns: {missing}')

df_sel = df.select(cols)
df_sel.describe()

KeyError: "Missing columns: ['afrr_activation_price_pos', 'afrr_activation_price_neg', 'afrr_activation_avg_price_pos', 'afrr_activation_avg_price_neg']"

In [11]:
# Count exact sentinel occurrences
sentinel_counts = df.select([
    (pl.col('afrr_activation_price_pos') == 99999.99).sum().alias('pos_99999_99'),
    (pl.col('afrr_activation_price_neg') == -99999.99).sum().alias('neg_99999_99'),
    (pl.col('afrr_activation_avg_price_pos') == 99999.99).sum().alias('avg_pos_99999_99'),
    (pl.col('afrr_activation_avg_price_neg') == -99999.99).sum().alias('avg_neg_99999_99'),
])
sentinel_counts

ColumnNotFoundError: unable to find column "afrr_activation_price_pos"; valid columns: ["timestamp_utc", "NRV_balance", "reBAP_shortage_surplus", "rz_saldo_mw", "afrr_activated_mw_pos", "afrr_activated_mw_neg", "mfrr_activated_mw_pos", "mfrr_activated_mw_neg", "afrr_activated_mwh_pos", "afrr_activated_mwh_neg", "mfrr_activated_mwh_pos", "mfrr_activated_mwh_neg", "activated_volume_pos_mw", "activated_volume_neg_mw", "activated_volume_pos_mwh", "activated_volume_neg_mwh"]

In [ ]:
# Skewness and Kurtosis for avg_price columns
skew = df.select([
    pl.col('afrr_activation_avg_price_pos').skew().alias('avg_price_pos_skew'),
    pl.col('afrr_activation_avg_price_neg').skew().alias('avg_price_neg_skew'),
])
kurt = df.select([
    pl.col('afrr_activation_avg_price_pos').kurtosis().alias('avg_price_pos_kurtosis'),
    pl.col('afrr_activation_avg_price_neg').kurtosis().alias('avg_price_neg_kurtosis'),
])
skew, kurt

## Teil 2: Realistische Werte (Referenz)

| Spalte | Typischer Bereich (Normal) | Stress / Krise | Verdächtig / Technisch |
|---|---|---|---|
| aFRR Avg Price (Pos) | 50 € bis 500 € / MWh | 1.000 € bis 10.000 € | > 90.000 € (Sentinel) |
| aFRR Avg Price (Neg) | -500 € bis 0 € / MWh | -5.000 € bis -1.000 € | < -90.000 € (Sentinel) |
| aFRR Activated MW | 0 bis 800 MW | 1.000 bis 2.500 MW | > 5.000 MW (physikalisch unmöglich) |
| aFRR Offered MW | 1.500 bis 4.000 MW | < 1.000 MW (Knappheit) | 0 MW (Datenfehler) |

Daumenregel: Ein Avg Price von 2.000 € ist extrem, aber möglich. Ein Activation Price von 99.999 € ist immer ein Platzhalter.

## Part 3: Price-Volume Logic Check

In [ ]:
# Rows with near-sentinel positive prices
cap_rows = df.filter(pl.col('afrr_activation_price_pos') > 90000)
cap_rows.height

In [ ]:
cap_stats = cap_rows.select(['afrr_activated_mw_pos']).describe()
overall_stats = df.select(['afrr_activated_mw_pos']).describe()
cap_stats, overall_stats

In [ ]:
cap_mean = cap_rows.select(pl.col('afrr_activated_mw_pos').mean()).item()
overall_mean = df.select(pl.col('afrr_activated_mw_pos').mean()).item()
print(f'Cap mean volume: {cap_mean:.2f} MW')
print(f'Overall mean volume: {overall_mean:.2f} MW')
print('Cap mean significantly lower:' , cap_mean < 0.5 * overall_mean)

## Additional Logic Checks (Teil 3)

In [ ]:
# Marginal vs Average consistency
consistency = df.select([
    (pl.col('afrr_activation_price_pos') >= pl.col('afrr_activation_avg_price_pos')).mean().alias('pos_price_ge_avg_ratio'),
    (pl.col('afrr_activation_price_neg') <= pl.col('afrr_activation_avg_price_neg')).mean().alias('neg_price_le_avg_ratio'),
])
consistency

In [ ]:
# Correlation with day-ahead price (if available)
if 'da_price_eur' in df.columns:
    corr_pos = df.select(pl.corr('da_price_eur', 'afrr_activation_price_pos')).item()
    corr_neg = df.select(pl.corr('da_price_eur', 'afrr_activation_price_neg')).item()
    print(f'Corr(DA, aFRR pos): {corr_pos:.3f}')
    print(f'Corr(DA, aFRR neg): {corr_neg:.3f}')
else:
    print('da_price_eur not available in this dataset')

In [ ]:
# Null-volume check: what price appears when volume == 0?
zero_vol = df.filter(pl.col('afrr_activated_mw_pos') == 0)
zero_vol.select(['afrr_activation_price_pos']).describe()